# Experimentos Univariados 1.1–1.8

**Tesis MEC** — Comparación TSFMs vs Modelos Clásicos bajo DGPs controlados  
**Setup:** T ∈ {200, 1000} | H = 24 | R = 500 | Semilla = 3649 | Modelos Core únicamente

---

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from mectesis.dgp import AR1, RandomWalk, AR1WithTrend, SeasonalDGP, AR1WithBreak
from mectesis.models import (
    ARIMAModel, ChronosModel,
    NaiveModel, DriftModel, SeasonalNaiveModel,
    SARIMAModel, ARIMAWithTrendModel, ARIMAWithBreakModel,
)
from mectesis.simulation import MonteCarloEngine

SEED   = int('03649')
H      = 24
R      = 500
T_LIST = [200, 1000]

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
pd.set_option("display.float_format", "{:.4f}".format)

print("Cargando Chronos-2 (puede tardar ~30 s)...")
chronos = ChronosModel(device="cpu")
print("Chronos-2 listo.")

In [ ]:
# ─── Funciones auxiliares ───────────────────────────────────────────────────

def run_exp(dgp, make_models_fn, dgp_params,
            T_list=T_LIST, H=H, n_sim=R, seed=SEED):
    """Corre MC para todos los T. make_models_fn(T) retorna lista de modelos."""
    all_results = {}
    for T in T_list:
        dgp.rng = np.random.default_rng(seed)  # reset para reproducibilidad
        models = make_models_fn(T)
        engine = MonteCarloEngine(dgp, models, seed=seed)
        print(f"  T={T}: {n_sim} reps × {len(models)} modelos...",
              end=" ", flush=True)
        all_results[T] = engine.run_monte_carlo(
            n_sim, T, H, dgp_params, verbose=False)
        print("OK")
    return all_results   # {T: {model_name: DataFrame}}


def compute_blocks(results_T):
    """Dado {model_name: df}, calcula promedios para h=1-12 y h=13-24."""
    out = {}
    for mname, df in results_T.items():
        df_h = df[df["horizon"] != "avg_all"].copy()
        df_h["horizon"] = df_h["horizon"].astype(int)
        out[mname] = {
            "h=1-12":  df_h[df_h["horizon"] <= 12].mean(numeric_only=True),
            "h=13-24": df_h[df_h["horizon"] >= 13].mean(numeric_only=True),
        }
    return out


def results_table(all_results):
    """Muestra tabla comparativa de métricas por bloque."""
    rows = []
    for T, res_T in sorted(all_results.items()):
        for mname, blk in compute_blocks(res_T).items():
            for bname, m in blk.items():
                rows.append({
                    "T": T, "Modelo": mname, "Bloque": bname,
                    "Bias":    round(float(m["bias"]),     4),
                    "Varianza":round(float(m["variance"]), 4),
                    "MSE":     round(float(m["mse"]),      4),
                    "RMSE":    round(float(m["rmse"]),     4),
                })
    df = pd.DataFrame(rows).set_index(["T", "Modelo", "Bloque"])
    display(df.style.format(precision=4)
              .background_gradient(subset=["RMSE"], cmap="YlOrRd"))


def plot_rep(dgp, make_models_fn, dgp_params,
             T=200, H=H, seed=SEED, title=""):
    """Visualización de una simulación representativa."""
    dgp_r = dgp.__class__(seed=seed)
    y = dgp_r.simulate(T=T, **dgp_params)
    y_train, y_test = y[:-H], y[-H:]
    models = make_models_fn(T)

    fig, ax = plt.subplots(figsize=(13, 4))
    x_tr = np.arange(len(y_train))
    x_te = np.arange(len(y_train), T)

    ax.plot(x_tr, y_train, color="steelblue", lw=1.4, alpha=0.85, label="Histórico")
    ax.plot(x_te, y_test, "k--", lw=1.5, label="Observado (test)")
    ax.axvline(len(y_train) - 0.5, color="gray", ls=":", lw=1, alpha=0.6)

    palette = ["crimson", "darkorange", "seagreen", "purple", "teal", "olive"]
    for i, m in enumerate(models):
        m.fit(y_train)
        ax.plot(x_te, m.forecast(H),
                color=palette[i % len(palette)],
                lw=1.5, marker="o", ms=3, label=m.name)

    ax.set(title=title, xlabel="t", ylabel="$Y_t$")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()


def plot_metrics(all_results, title="", metrics=("rmse", "bias")):
    """Gráficos de métricas vs horizonte h=1..24 por modelo."""
    T_list = sorted(all_results.keys())
    fig, axes = plt.subplots(
        len(metrics), len(T_list),
        figsize=(7 * len(T_list), 3.5 * len(metrics)),
        squeeze=False,
    )
    palette = ["crimson", "darkorange", "seagreen", "purple", "teal", "steelblue"]

    for col, T in enumerate(T_list):
        for row, metric in enumerate(metrics):
            ax = axes[row][col]
            for i, (mname, df) in enumerate(all_results[T].items()):
                df_h = df[df["horizon"] != "avg_all"].copy()
                df_h["horizon"] = df_h["horizon"].astype(int)
                ax.plot(df_h["horizon"], df_h[metric],
                        label=mname, color=palette[i % len(palette)], lw=1.5)
            ax.axvline(12.5, color="gray", ls=":", lw=0.8, alpha=0.5)
            ax.set(
                title=f"T={T} — {metric.upper()}",
                xlabel="Horizonte h",
                ylabel=metric.upper(),
            )
            ax.legend(fontsize=8)

    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

---
## Experimento 1.1

**DGP:** AR(1) baja persistencia — $Y_t = 0.3\,Y_{t-1} + \varepsilon_t$  
**Modelos core:** ARIMA(1,0,0), Naive, Drift, Chronos-2

In [ ]:
dgp_1_1         = AR1(seed=SEED)
make_models_1_1 = lambda T: [ARIMAModel((1,0,0)), NaiveModel(), DriftModel(), chronos]
dgp_params_1_1  = {'phi': 0.3}

print("\n=== Experimento 1.1 ===")
results_1_1 = run_exp(
    dgp_1_1,
    make_models_1_1,
    dgp_params_1_1,
)

In [ ]:
# Visualización representativa (T=200)
plot_rep(
    dgp_1_1, make_models_1_1, dgp_params_1_1,
    T=200, title="Exp 1.1 — Simulación representativa (T=200, seed=SEED)"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.1")
results_table(results_1_1)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_1,
    title="Exp 1.1 — Métricas por horizonte (R={})".format(R)
)

---
## Experimento 1.2

**DGP:** AR(1) alta persistencia — $Y_t = 0.9\,Y_{t-1} + \varepsilon_t$  
**Modelos core:** ARIMA(1,0,0), Naive, Chronos-2

In [ ]:
dgp_1_2         = AR1(seed=SEED)
make_models_1_2 = lambda T: [ARIMAModel((1,0,0)), NaiveModel(), chronos]
dgp_params_1_2  = {'phi': 0.9}

print("\n=== Experimento 1.2 ===")
results_1_2 = run_exp(
    dgp_1_2,
    make_models_1_2,
    dgp_params_1_2,
)

In [ ]:
# Visualización representativa (T=200)
plot_rep(
    dgp_1_2, make_models_1_2, dgp_params_1_2,
    T=200, title="Exp 1.2 — Simulación representativa (T=200, seed=SEED)"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.2")
results_table(results_1_2)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_2,
    title="Exp 1.2 — Métricas por horizonte (R={})".format(R)
)

---
## Experimento 1.3

**DGP:** Random Walk I(1) sin drift — $Y_t = Y_{t-1} + \varepsilon_t$  
**Modelos core:** ARIMA(0,1,0), Drift, Chronos-2

In [ ]:
dgp_1_3         = RandomWalk(seed=SEED)
make_models_1_3 = lambda T: [ARIMAModel((0,1,0)), DriftModel(), chronos]
dgp_params_1_3  = {'drift': 0.0}

print("\n=== Experimento 1.3 ===")
results_1_3 = run_exp(
    dgp_1_3,
    make_models_1_3,
    dgp_params_1_3,
)

In [ ]:
# Visualización representativa (T=200)
plot_rep(
    dgp_1_3, make_models_1_3, dgp_params_1_3,
    T=200, title="Exp 1.3 — Simulación representativa (T=200, seed=SEED)"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.3")
results_table(results_1_3)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_3,
    title="Exp 1.3 — Métricas por horizonte (R={})".format(R)
)

---
## Experimento 1.4

**DGP:** Random Walk I(1) con drift — $Y_t = 0.5 + Y_{t-1} + \varepsilon_t$  
**Modelos core:** ARIMA(0,1,0), Drift, Chronos-2

In [ ]:
dgp_1_4         = RandomWalk(seed=SEED)
make_models_1_4 = lambda T: [ARIMAModel((0,1,0)), DriftModel(), chronos]
dgp_params_1_4  = {'drift': 0.5}

print("\n=== Experimento 1.4 ===")
results_1_4 = run_exp(
    dgp_1_4,
    make_models_1_4,
    dgp_params_1_4,
)

In [ ]:
# Visualización representativa (T=200)
plot_rep(
    dgp_1_4, make_models_1_4, dgp_params_1_4,
    T=200, title="Exp 1.4 — Simulación representativa (T=200, seed=SEED)"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.4")
results_table(results_1_4)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_4,
    title="Exp 1.4 — Métricas por horizonte (R={})".format(R)
)

---
## Experimento 1.5

**DGP:** AR(1) + tendencia — $Y_t = 5 + 0.1t + 0.6\,Y_{t-1} + \varepsilon_t$  
**Modelos core:** ARIMA(1,0,0)+trend (trend='ct'), Chronos-2

In [ ]:
dgp_1_5         = AR1WithTrend(seed=SEED)
make_models_1_5 = lambda T: [ARIMAWithTrendModel((1,0,0), trend='ct'), chronos]
dgp_params_1_5  = {'intercept': 5.0, 'trend_coeff': 0.1, 'phi': 0.6}

print("\n=== Experimento 1.5 ===")
results_1_5 = run_exp(
    dgp_1_5,
    make_models_1_5,
    dgp_params_1_5,
)

In [ ]:
# Visualización representativa (T=200)
plot_rep(
    dgp_1_5, make_models_1_5, dgp_params_1_5,
    T=200, title="Exp 1.5 — Simulación representativa (T=200, seed=SEED)"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.5")
results_table(results_1_5)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_5,
    title="Exp 1.5 — Métricas por horizonte (R={})".format(R)
)

---
## Experimento 1.6

**DGP:** SARIMA trimestral (s=4) — $(1-0.5L)(1-0.3L^4)Y_t = \varepsilon_t$  
**Modelos core:** SARIMA(1,0,0)(1,0,0)_4, SeasonalNaive(s=4), Chronos-2

In [ ]:
dgp_1_6         = SeasonalDGP(seed=SEED)
make_models_1_6 = lambda T: [SARIMAModel((1,0,0),(1,0,0,4)), SeasonalNaiveModel(4), chronos]
dgp_params_1_6  = {'phi': 0.5, 'Phi': 0.3, 's': 4, 'integrated': False}

print("\n=== Experimento 1.6 ===")
results_1_6 = run_exp(
    dgp_1_6,
    make_models_1_6,
    dgp_params_1_6,
)

In [ ]:
# Visualización representativa (T=200)
plot_rep(
    dgp_1_6, make_models_1_6, dgp_params_1_6,
    T=200, title="Exp 1.6 — Simulación representativa (T=200, seed=SEED)"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.6")
results_table(results_1_6)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_6,
    title="Exp 1.6 — Métricas por horizonte (R={})".format(R)
)

---
## Experimento 1.7

**DGP:** SARIMA mensual (s=12) — $(1-L)(1-L^{12})Y_t = \varepsilon_t$  
**Modelos core:** SARIMA(0,1,0)(0,1,0)_12, SeasonalNaive(s=12), Chronos-2

In [ ]:
dgp_1_7         = SeasonalDGP(seed=SEED)
make_models_1_7 = lambda T: [SARIMAModel((0,1,0),(0,1,0,12)), SeasonalNaiveModel(12), chronos]
dgp_params_1_7  = {'integrated': True, 's': 12}

print("\n=== Experimento 1.7 ===")
results_1_7 = run_exp(
    dgp_1_7,
    make_models_1_7,
    dgp_params_1_7,
)

In [ ]:
# Visualización representativa (T=200)
plot_rep(
    dgp_1_7, make_models_1_7, dgp_params_1_7,
    T=200, title="Exp 1.7 — Simulación representativa (T=200, seed=SEED)"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.7")
results_table(results_1_7)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_7,
    title="Exp 1.7 — Métricas por horizonte (R={})".format(R)
)

---
## Experimento 1.8

**DGP:** AR(1) con quiebre estructural en $T/2$ — $\phi$ cambia de 0.3 a 0.8  
**Modelos core:** ARIMA(1,0,0)+break (dummy exógena), Chronos-2

In [ ]:
dgp_1_8         = AR1WithBreak(seed=SEED)
make_models_1_8 = lambda T: [ARIMAWithBreakModel((1,0,0), T_total=T), chronos]
dgp_params_1_8  = {'phi_before': 0.3, 'phi_after': 0.8}

print("\n=== Experimento 1.8 ===")
results_1_8 = run_exp(
    dgp_1_8,
    make_models_1_8,
    dgp_params_1_8,
)

In [ ]:
# Visualización representativa (T=200)
plot_rep(
    dgp_1_8, make_models_1_8, dgp_params_1_8,
    T=200, title="Exp 1.8 — Simulación representativa (T=200, seed=SEED)"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.8")
results_table(results_1_8)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_8,
    title="Exp 1.8 — Métricas por horizonte (R={})".format(R)
)